# Creating Synthetic Observations with perfect_model_obs

Sample a model state as if it were the real ocean -  the first step in an Observing System Simulation Experiment (OSSE)!

A typical synthetic-observation workflow consists these steps:

1. Design an observing network: the locations from Tutorial 1, plus some random ones.
2. Run a "truth" or nature run of your model - MOM6 for this tutorial
3. Run `perfect_model_obs` through `dartobsgen` to create the synthetic observations
4. Examine the synthetic observations you have created. 

*This is Part 2 of the DART tutorial series:*   
[1. Working with Real Observations](tutorial1_real_observations.ipynb) ·
**2. Creating Synthetic Observations** ·
[3. Cycling DART–CESM](tutorial3_cycling_dart_cesm.ipynb)

```{admonition} What you'll learn
:class: tip

- Why (and when) synthetic observations are useful.
- What `perfect_model_obs` does to generate synthetic observations: 
  apply DART's forward operators to a known "truth" run of the model (MOM6)
  and add noise drawn from the observation error.
- Why OSSEs (Observing System Simulation Experiments) are a way to test a DA system before
  trusting real-observation results
- What to think about when designing an observing network programmatically 
```

```{admonition} What you'll produce
:class: important

A directory `obs/synthetic/` of obs_seq files sampled from a MOM6 run at the Tutorial 1
observation locations and times **plus 20 random profile locations**. The synthetic files are a
drop-in replacement for the real ones in Tutorial 3 - that swap turns Tutorial 3 into an
OSSE.
```

````{admonition} Missed Tutorial 1?
:class: dropdown

You can run this notebook standalone:

- Point `REAL_OBS_DIR` at the staged workshop copy, `<CROC_DART_OBS>/real`, in Step 3.1, **or**
- Skip the harvest entirely. Step 3.1 falls back to a regular-grid network if it finds
  no Tutorial 1 output.
````

# SECTION 1: Why synthetic observations?

In an **OSSE** you treat one model run as the "truth". You sample your model run where your
instruments would sample the real ocean, and add realistic errors such as instrument noise or 
representativeness error. Because you know the truth exactly, you can use OSSEs to estimate 
how new instruments and observing networks will impact your data assimilation. 

OSSEs answer questions like:

- Is my DA plumbing working at all? (If you can't recover a known truth, real obs won't help.)
- Where should new instruments go to constrain the circulation feature I care about?
- How does observation accuracy trade off against observation count?

`perfect_model_obs` is DART's tool for the sampling step: it reads a model state, applies
the same forward operators the filter would use, and perturbs each value with
noise drawn from the observation error variance you assign.

```{admonition} The identical-twin caveat
:class: note

Sampling a model with the same model that assimilates the samples ("identical twin"
experiments) gives optimistic results. The model error is zero by construction. Fine for
testing plumbing and observing-network design; be careful generalizing skill estimates to
the real ocean.
```

# SECTION 2: Build DART for MOM6


## Step 2.1: Compile perfect_model_obs

You cloned DART in Tutorial 1 ("Before you start"). Now build the MOM6 executables on
Derecho: load the same compiler and netCDF modules you use for CESM, then:

```bash
cd <DART_SRC>/models/MOM6/work
./quickbuild.sh
```

This compiles `perfect_model_obs`, `filter`, and the supporting utilities into the `work`
directory. See the [DART getting-started docs](https://docs.dart.ucar.edu) for
Derecho-specific build settings (`mkmf.template`).

## Step 2.2: What perfect_model_obs needs

`dartobsgen`'s `PerfectModelSource` drives `perfect_model_obs` for you, one assimilation
window at a time. It needs three things in the DART work directory:

1. the compiled `perfect_model_obs` executable,
2. an `input.nml` containing a `&perfect_model_obs_nml` block (the one shipped in
   `models/MOM6/work` is a good starting point), and
3. a **MOM6 state to sample**, the "truth". Edit `input.nml` so the model reads the
   staged panama restart, `<CROC_DART_OBS>/panama1_restart.nc` (copy it into the work
   directory first).

For each window, `PerfectModelSource` writes a template `obs_seq.in` holding your network
(locations, types, times, error variances, with placeholder values of 0.0), patches the
namelist, runs the executable in an isolated subdirectory under
`<DART_SRC>/models/MOM6/work/windows/`, and collects the resulting `obs_seq.out`.

## Step 2.3: Parameters and sanity check

The shared series parameters, then a quick check that the pieces from Steps 2.1–2.2 are
in place.

In [ ]:
# --- CROCODILE DART tutorial series parameters (same cell in all 3 notebooks) ---
from pathlib import Path
import datetime

DA_PROJECT_DIR = Path("<DART_DA_PROJECT_DIR>")        # your scratch working directory on Derecho

START = datetime.datetime(2013, 4, 1)   # must match RUN_STARTDATE in Tutorial 3
END   = datetime.datetime(2013, 4, 4)   # 3 days -> 3 one-day assimilation windows, centered on midnight
FREQ  = datetime.timedelta(hours=24)

# Bounding box: the panama1 domain (lon 278-281E, lat 7-10N) padded by ~2 degrees
LAT_MIN, LAT_MAX = 5.0, 12.0
LON_MIN, LON_MAX = -84.0, -77.0

OBS_TYPES = ["ARGO_TEMPERATURE", "ARGO_SALINITY"]

REAL_OBS_DIR      = DA_PROJECT_DIR / "obs" / "real"       # Tutorial 1 output
SYNTHETIC_OBS_DIR = DA_PROJECT_DIR / "obs" / "synthetic"  # Tutorial 2 output

In [ ]:
DART_WORK_DIR = Path("<DART_SRC>") / "models" / "MOM6" / "work"

for required in ["perfect_model_obs", "input.nml"]:
    f = DART_WORK_DIR / required
    status = "OK     " if f.exists() else "MISSING"
    print(f"{status} {f}")

# SECTION 3: Design the observation network

## Step 3.1: Harvest the Tutorial 1 locations

A realistic OSSE samples the truth where instruments actually were. We read the first
Tutorial 1 window with pyDARTdiags and turn its unique locations into `ObsNetworkEntry`
objects, one per location per observation type.

One thing changes compared to Tutorial 1: the **error variance is now yours to choose**.
Real converters carry the instrument error with the data; in an OSSE the "instrument" is
imaginary, so you decide how good it is. We use (0.2&nbsp;°C)² for temperature and
(0.1&nbsp;PSU)² for salinity, typical Argo-like values.

In [ ]:
import numpy as np
import pydartdiags.obs_sequence.obs_sequence as obsq
from dartobsgen import ObsNetworkEntry

OBS_ERR_VAR = {
    "ARGO_TEMPERATURE": 0.04,  # (0.2 degC)^2
    "ARGO_SALINITY":    0.01,  # (0.1 PSU)^2
}
PROFILE_DEPTHS = [10.0, 50.0, 100.0, 200.0, 500.0, 1000.0]  # Argo-like profile depths (m)

nb1_files = sorted(REAL_OBS_DIR.glob("obs_seq.*.out"))
# Missed Tutorial 1? Use the staged workshop copy instead:
# nb1_files = sorted(Path("<CROC_DART_OBS>/real").glob("obs_seq.*.out"))

network = []
if nb1_files:
    real = obsq.ObsSequence(str(nb1_files[0]))
    locations = (real.df[["longitude", "latitude", "vertical"]]
                 .drop_duplicates()
                 .reset_index(drop=True))
    print(f"Harvested {len(locations)} unique locations from {nb1_files[0].name}")
    for _, row in locations.iterrows():
        lon = row.longitude if row.longitude <= 180 else row.longitude - 360
        for obs_type in OBS_TYPES:
            network.append(ObsNetworkEntry(
                obs_type=obs_type,
                lat=float(row.latitude),
                lon=float(lon),
                vertical=float(row.vertical),
                vert_unit="height (m)",
                obs_err_var=OBS_ERR_VAR[obs_type],
            ))
else:
    # Fallback: no Tutorial 1 output found -- build a coarse regular-grid network.
    print("No Tutorial 1 output found; building a 1-degree grid network instead.")
    for lat in np.arange(LAT_MIN + 0.5, LAT_MAX, 1.0):
        for lon in np.arange(LON_MIN + 0.5, LON_MAX, 1.0):
            for depth in PROFILE_DEPTHS:
                for obs_type in OBS_TYPES:
                    network.append(ObsNetworkEntry(
                        obs_type=obs_type,
                        lat=float(lat), lon=float(lon),
                        vertical=depth, vert_unit="height (m)",
                        obs_err_var=OBS_ERR_VAR[obs_type],
                    ))

n_harvested = len(network)
print(f"Network so far: {n_harvested} observations")

## Step 3.2: Add random locations

Now the part you can't do with real data: **invent instruments**. We add 20 random
profile locations inside the domain, each sampling the standard depths. The seeded random
generator means everyone in the workshop gets the same "random" network; change the seed
and your network is yours alone.

In [ ]:
rng = np.random.default_rng(seed=42)   # fixed seed: reproducible "random" network
N_RANDOM = 20

random_lats = rng.uniform(LAT_MIN, LAT_MAX, N_RANDOM)
random_lons = rng.uniform(LON_MIN, LON_MAX, N_RANDOM)

for lat, lon in zip(random_lats, random_lons):
    for depth in PROFILE_DEPTHS:
        for obs_type in OBS_TYPES:
            network.append(ObsNetworkEntry(
                obs_type=obs_type,
                lat=float(lat), lon=float(lon),
                vertical=depth, vert_unit="height (m)",
                obs_err_var=OBS_ERR_VAR[obs_type],
            ))

print(f"Added {len(network) - n_harvested} obs at {N_RANDOM} random profile locations "
      f"({len(network)} total)")

## Step 3.3: Map the network

Plot the two populations, the locations harvested from Tutorial 1 and the random
additions, together.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

net_df = pd.DataFrame([{
    "obs_type": e.obs_type, "lat": e.lat, "lon": e.lon,
    "vertical": e.vertical, "obs_err_var": e.obs_err_var,
} for e in network])

harvested = net_df.iloc[:n_harvested]
random_part = net_df.iloc[n_harvested:]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(harvested["lon"], harvested["lat"], s=18, color="tab:blue",
           label=f"Tutorial 1 locations ({len(harvested)})")
ax.scatter(random_part["lon"], random_part["lat"], s=30, color="tab:orange",
           marker="^", label=f"random additions ({len(random_part)})")
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Synthetic observing network")
ax.legend()
plt.show()

````{admonition} Try it: a targeted observing campaign
:class: attention

Instead of scattering the 20 random profiles over the whole domain, cluster them inside a
1°×1° sub-box (change the bounds passed to `rng.uniform`). Regenerate the map. Where would
*you* put floats to constrain the coastal current along the Panama shelf, and what does
that choice cost you elsewhere in the domain?
````

````{admonition} Hint
:class: dropdown

```python
random_lats = rng.uniform(8.0, 9.0, N_RANDOM)
random_lons = rng.uniform(-80.0, -79.0, N_RANDOM)
```

Dense local sampling constrains the feature you target, but the rest of the domain is
corrected only through ensemble covariances, and in Tutorial 3, localization deliberately
limits how far one observation can reach.
````

# SECTION 4: Generate the synthetic observations

## Step 4.1: Run perfect_model_obs through dartobsgen

Same `ObsGenConfig` pattern as Tutorial 1; only the source changes. The source for "observations"
is now a MOM6 run rather than CrocoLake. 



In [ ]:
from dartobsgen import ObsGenConfig, PerfectModelSource, generate_obs_sequences

config = ObsGenConfig(
    start=START, end=END,
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
    obs_types=OBS_TYPES,
    assimilation_frequency=FREQ,
    output_dir=SYNTHETIC_OBS_DIR,
)

source = PerfectModelSource(
    dart_work_dir=str(DART_WORK_DIR),
    obs_network=network,
)

SYNTHETIC_OBS_DIR.mkdir(parents=True, exist_ok=True)

# max_workers=1 runs windows sequentially; set to None to run them in parallel.
written = generate_obs_sequences(config, source, max_workers=1)

print(f"{len(written)} obs_seq file(s) written to {SYNTHETIC_OBS_DIR.name}/")
for p in written:
    print("  ", Path(p).name)

## Step 4.2: Look inside

The template values (0.0) have been replaced by the model sampled at each location, plus
noise drawn from your `obs_err_var`.

In [ ]:
syn = obsq.ObsSequence(str(Path(written[0])))

print(f"{Path(written[0]).name}: {len(syn.df)} observations")
syn.df[["type", "longitude", "latitude", "vertical", "observation", "obs_err_var"]].head(8)

## Step 4.3: Map every window

Loop over every obs_seq file `generate_obs_sequences` wrote and map the
`ARGO_TEMPERATURE` observations it holds, colored by the value `perfect_model_obs`
sampled. One panel per assimilation window, all sharing a color scale, so you can watch
the sampled field itself evolve across the 3-day run.

In [ ]:
temp_dfs = []
for p in written:
    obs_seq = obsq.ObsSequence(str(Path(p)))
    temp_dfs.append(obs_seq.df[obs_seq.df["type"] == "ARGO_TEMPERATURE"])

vmin = min(df["observation"].min() for df in temp_dfs)
vmax = max(df["observation"].max() for df in temp_dfs)

ncols = 4
nrows = -(-len(written) // ncols)  # ceil division
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows),
                          sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, p, df in zip(axes, written, temp_dfs):
    lon = np.where(df["longitude"] > 180, df["longitude"] - 360, df["longitude"])
    sc = ax.scatter(lon, df["latitude"], c=df["observation"],
                     cmap="viridis", vmin=vmin, vmax=vmax, s=14)
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_title(Path(p).name.removeprefix("obs_seq.").removesuffix(".out"), fontsize=8)

for ax in axes[len(written):]:
    ax.axis("off")

fig.colorbar(sc, ax=axes[:len(written)].tolist(), shrink=0.6,
             label="ARGO_TEMPERATURE (degC)")
fig.suptitle("perfect_model_obs temperature, every assimilation window")
plt.show()

# Recap

```{admonition} What you learned
:class: tip

- `perfect_model_obs` samples a known model state with the same forward operators the
  filter uses, adding noise from the observation error variance.
- An observing network is just a list of (type, location, depth, error variance) entries:
  `ObsNetworkEntry`, and you can build it from real locations, random draws, or any
  design you can code.
- In an OSSE, **error variance is a design choice**.
```

**Your takeaway artifact:** A directory `obs/synthetic/` containing synthetic observations 
under your `DA_PROJECT_DIR`.

# Where to go from here?

- **[Tutorial 3: Cycling DART–CESM](tutorial3_cycling_dart_cesm.ipynb)**: assimilate the
  real observations, then swap in `obs/synthetic/` to run it as an OSSE.
- The script version of this workflow:
  [`mom6_perfect_model.py`](https://github.com/CROCODILE-CESM/dartobsgen) in the
  dartobsgen repository.
- [DART perfect_model_obs documentation](https://docs.dart.ucar.edu/en/latest/assimilation_code/programs/perfect_model_obs/perfect_model_obs.html).